Stacks and queues are the two simplest abstract data structures — and among the most useful. They are not new storage mechanisms; they are arrays or linked lists with restricted access rules. That restriction is the whole point: by controlling *where* you can insert and remove, you get a structure whose behaviour is predictable and easy to reason about. Stacks power function calls, undo systems, and depth-first search. Queues power task scheduling, breadth-first search, and streaming pipelines.

## Stack — Last In, First Out

A **stack** allows access at only one end — the **top**. The rule is simple: the last element pushed onto the stack is the first one popped off. This is called **LIFO** — Last In, First Out.

Think of a stack of plates. You place a new plate on top; you take a plate from the top. You never pull one from the middle.

![Stack Operations](https://raw.githubusercontent.com/schemabotview/dsa/main/img/stack-operations.svg)

All four core operations are **O(1)**:
- `push(x)` — add element to the top
- `pop()` — remove and return the top element
- `peek()` — return the top element without removing it
- `is_empty()` — check if the stack is empty

## Stack Implementation

### Python

In [ ]:
# Python's list is a perfectly good stack.
# append() = push  (O(1) amortised)
# pop()    = pop   (O(1))
# [-1]     = peek  (O(1))

stack = []
stack.append(10)   # push
stack.append(30)
stack.append(50)

print(stack[-1])   # peek → 50
print(stack.pop()) # pop  → 50
print(stack.pop()) # pop  → 30
print(stack)       # [10]

### Kotlin

```kotlin
fun main() {
    // ArrayDeque used as a stack — all operations O(1)
    val stack = ArrayDeque<Int>()

    stack.addLast(10)   // push
    stack.addLast(30)
    stack.addLast(50)

    println(stack.last())       // peek → 50
    println(stack.removeLast()) // pop  → 50
    println(stack.removeLast()) // pop  → 30
    println(stack)              // [10]
}
```

## Stack Use Case — Balanced Parentheses

Given a string containing `()`, `[]`, and `{}`, determine whether every opening bracket has a matching closing bracket in the correct order. This is a canonical stack problem — open brackets are pushed; each closing bracket must match the most recently opened one (the top of the stack).

### Python

In [ ]:
def is_balanced(s: str) -> bool:
    match = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in '([{':
            stack.append(ch)              # push open bracket
        elif ch in ')]}':
            if not stack or stack[-1] != match[ch]:
                return False              # no match — unbalanced
            stack.pop()                   # matched — pop
    return len(stack) == 0                # stack must be empty at the end


print(is_balanced("({[]})"))   # True
print(is_balanced("({)}"))     # False — wrong order
print(is_balanced("((("))      # False — unclosed
print(is_balanced(""))         # True  — empty string is balanced

### Kotlin

```kotlin
fun isBalanced(s: String): Boolean {
    val match = mapOf(')' to '(', ']' to '[', '}' to '{')
    val stack = ArrayDeque<Char>()
    for (ch in s) {
        when {
            ch in "([{" -> stack.addLast(ch)
            ch in ")]}" -> {
                if (stack.isEmpty() || stack.last() != match[ch]) return false
                stack.removeLast()
            }
        }
    }
    return stack.isEmpty()
}
```

## Stack Use Case — Evaluating Expressions (Monotonic Stack)

A **monotonic stack** maintains elements in sorted order (increasing or decreasing) by popping elements that violate the order before pushing the new one. This pattern solves a whole family of problems: next greater element, daily temperatures, largest rectangle in histogram.

**Next greater element** — for each element, find the next element to its right that is larger.

### Python

In [ ]:
def next_greater(nums: list[int]) -> list[int]:
    """
    For each element, find the next element to its right that is greater.
    Returns -1 if none exists.  O(n) time — each element pushed/popped once.
    """
    result = [-1] * len(nums)
    stack  = []            # stores indices of elements waiting for their answer

    for i, val in enumerate(nums):
        # val is the next greater element for everything in stack that is smaller
        while stack and nums[stack[-1]] < val:
            idx = stack.pop()
            result[idx] = val
        stack.append(i)

    return result


print(next_greater([2, 1, 5, 3, 4]))
# [5, 5, -1, 4, -1]
# 2 → 5,  1 → 5,  5 → none,  3 → 4,  4 → none

### Kotlin

```kotlin
fun nextGreater(nums: IntArray): IntArray {
    val result = IntArray(nums.size) { -1 }
    val stack  = ArrayDeque<Int>()   // stores indices

    for (i in nums.indices) {
        while (stack.isNotEmpty() && nums[stack.last()] < nums[i]) {
            result[stack.removeLast()] = nums[i]
        }
        stack.addLast(i)
    }
    return result
}
```

## Queue — First In, First Out

A **queue** allows insertions at one end (the **rear**) and removals from the other end (the **front**). The rule: the first element inserted is the first one removed. This is called **FIFO** — First In, First Out.

Think of a checkout line at a shop. New customers join the back; the cashier serves the person at the front. Nobody cuts in.

![Queue Operations](https://raw.githubusercontent.com/schemabotview/dsa/main/img/queue-operations.svg)

All four core operations are **O(1)**:
- `enqueue(x)` — add element at the rear
- `dequeue()` — remove and return the element at the front
- `peek()` — return the front element without removing it
- `is_empty()` — check if the queue is empty

> **Never use a plain `list` as a queue in Python.** `list.pop(0)` is O(n) — it shifts every remaining element. Use `collections.deque` which gives O(1) `popleft()`.

## Queue Implementation

### Python

In [ ]:
from collections import deque

queue = deque()
queue.append(10)     # enqueue at rear
queue.append(30)
queue.append(50)

print(queue[0])           # peek front → 10
print(queue.popleft())    # dequeue    → 10  (O(1))
print(queue.popleft())    # dequeue    → 30
print(queue)              # deque([50])

### Kotlin

```kotlin
fun main() {
    // ArrayDeque used as a queue
    val queue = ArrayDeque<Int>()

    queue.addLast(10)     // enqueue
    queue.addLast(30)
    queue.addLast(50)

    println(queue.first())         // peek → 10
    println(queue.removeFirst())   // dequeue → 10
    println(queue.removeFirst())   // dequeue → 30
    println(queue)                 // [50]
}
```

## Queue Use Case — BFS Level-Order Traversal

A queue is the natural structure for **breadth-first search** because BFS processes nodes level by level — all nodes at distance 1 before distance 2, distance 2 before distance 3, and so on. A queue enforces exactly this order: nodes are processed in the order they were discovered (FIFO).

Here is BFS on a simple graph. Full coverage of BFS and DFS comes in notebook 14.

### Python

In [ ]:
from collections import deque

def bfs(graph: dict[int, list[int]], start: int) -> list[int]:
    """Return nodes in BFS (level-order) visit order."""
    visited = {start}
    queue   = deque([start])
    order   = []

    while queue:
        node = queue.popleft()       # process front of queue
        order.append(node)
        for neighbour in graph[node]:
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(neighbour)  # discover → enqueue

    return order


graph = {
    1: [2, 3],
    2: [4, 5],
    3: [6],
    4: [], 5: [], 6: []
}
print(bfs(graph, 1))   # [1, 2, 3, 4, 5, 6]  — level by level

### Kotlin

```kotlin
fun bfs(graph: Map<Int, List<Int>>, start: Int): List<Int> {
    val visited = mutableSetOf(start)
    val queue   = ArrayDeque<Int>().also { it.addLast(start) }
    val order   = mutableListOf<Int>()

    while (queue.isNotEmpty()) {
        val node = queue.removeFirst()
        order.add(node)
        for (neighbour in graph[node] ?: emptyList()) {
            if (neighbour !in visited) {
                visited.add(neighbour)
                queue.addLast(neighbour)
            }
        }
    }
    return order
}
```

## Deque — Double-Ended Queue

A **deque** (pronounced *deck*) generalises both stack and queue by allowing O(1) push and pop from *both* ends. Python's `collections.deque` and Kotlin's `ArrayDeque` are both deques.

| Operation | deque |
|---|---|
| Push/pop at rear | O(1) |
| Push/pop at front | O(1) |
| Peek at either end | O(1) |
| Access by index | O(n) |

A classic deque pattern is the **sliding window maximum** — maintaining the maximum value in a window as it slides across an array in O(n) total using a monotonic deque.

### Python

In [ ]:
from collections import deque

def sliding_window_max(nums: list[int], k: int) -> list[int]:
    """
    Return the maximum of each window of size k.
    O(n) — each element is added/removed from the deque at most once.
    The deque stores indices and is kept in decreasing order of values.
    """
    dq     = deque()   # stores indices; front is always the window's max
    result = []

    for i, val in enumerate(nums):
        # Remove indices that have slid out of the window
        if dq and dq[0] <= i - k:
            dq.popleft()

        # Maintain decreasing order — pop smaller elements from rear
        while dq and nums[dq[-1]] < val:
            dq.pop()

        dq.append(i)

        if i >= k - 1:             # window is fully formed
            result.append(nums[dq[0]])

    return result


print(sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], k=3))
# [3, 3, 5, 5, 6, 7]

### Kotlin

```kotlin
fun slidingWindowMax(nums: IntArray, k: Int): IntArray {
    val dq     = ArrayDeque<Int>()    // stores indices
    val result = mutableListOf<Int>()

    for (i in nums.indices) {
        if (dq.isNotEmpty() && dq.first() <= i - k) dq.removeFirst()
        while (dq.isNotEmpty() && nums[dq.last()] < nums[i]) dq.removeLast()
        dq.addLast(i)
        if (i >= k - 1) result.add(nums[dq.first()])
    }
    return result.toIntArray()
}
```

## Stack vs Queue — Summary

| | Stack | Queue |
|---|---|---|
| Access rule | LIFO — last in, first out | FIFO — first in, first out |
| Insert | push at top | enqueue at rear |
| Remove | pop from top | dequeue from front |
| Typical uses | DFS, undo/redo, call stack, balanced brackets | BFS, task scheduling, sliding window |
| Python | `list` (append/pop) | `collections.deque` (append/popleft) |
| Kotlin | `ArrayDeque` (addLast/removeLast) | `ArrayDeque` (addLast/removeFirst) |
| All core ops | O(1) | O(1) |

## Key Takeaways

- A **stack** is LIFO — push and pop happen at the same end (the top). All core operations are O(1). In Python use `list`; in Kotlin use `ArrayDeque` with `addLast`/`removeLast`.
- A **queue** is FIFO — enqueue at the rear, dequeue from the front. All core operations are O(1). In Python use `collections.deque`; **never `list.pop(0)`** which is O(n).
- A **deque** is the generalisation — O(1) at both ends. It can act as a stack, a queue, or both.
- **Balanced parentheses** is the canonical stack problem — push opens, pop and match closes.
- **Monotonic stack** — pop elements that violate the sort order before pushing — solves next greater/smaller element problems in O(n).
- **BFS uses a queue** — nodes are discovered and processed in FIFO order, guaranteeing level-by-level traversal.
- **Monotonic deque** — sliding window maximum in O(n) by maintaining decreasing order and evicting out-of-window indices.